In [1]:
#!py -m pip install -r ../requirements.txt

In [2]:
%load_ext autoreload
%autoreload 2

import json
import pathlib
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm.auto import tqdm

from src.dns_utils import resolve_dns, to_apex, is_conclusive

IN_CSV = "../data/interim/01_data_after_yfinance.csv"
RAW = pathlib.Path("../data/interim/02_dns_raw.jsonl")
RAW.parent.mkdir(exist_ok=True)

MAX_WORKERS = 10   # 1 = sequencial; suba aos poucos olhando a taxa de timeout

C:\Users\felip\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv(IN_CSV)
df["apex"] = df["website"].map(to_apex)

print("sem domínio válido:", df["apex"].isna().sum())
domains = df["apex"].dropna().unique()
print("domínios únicos:", len(domains))

sem domínio válido: 343
domínios únicos: 10261


In [4]:
done = set()
if RAW.exists():
    with RAW.open(encoding="utf-8") as f:
        done = {r["domain"] for r in map(json.loads, f) if is_conclusive(r)}

todo = [d for d in domains if d not in done]
print(f"já feitos: {len(done)} | a fazer: {len(todo)}")

já feitos: 10220 | a fazer: 42


In [5]:
def safe_resolve(d):
    """Rede de segurança: bug inesperado vira registro de erro, não derruba o loop."""
    try:
        return resolve_dns(d)
    except Exception as e:
        return {"domain": d, "apex": None, "NS": f"error: {e}",
                "NS_apex": None, "NS_A": None, "NS_AAAA": None}

# Só a thread principal escreve no arquivo (evita corrida de escrita).
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool, RAW.open("a", encoding="utf-8") as f:
    futures = [pool.submit(safe_resolve, d) for d in todo]
    for fut in tqdm(as_completed(futures), total=len(futures)):
        f.write(json.dumps(fut.result(), ensure_ascii=False) + "\n")
        f.flush()  # grava no disco agora: se o kernel cair, nada se perde

100%|██████████| 42/42 [00:21<00:00,  1.96it/s]


In [6]:
raw = pd.read_json(RAW, lines=True).drop_duplicates("domain", keep="last")

is_list = raw["NS"].map(lambda x: isinstance(x, list))

# 1 linha por domínio sem NS (nxdomain, timeout, invalid...)
dns_status = raw.loc[~is_list, ["domain", "NS"]].rename(columns={"NS": "status"})

# 1 linha por domínio × NS
ns_hosts = (raw[is_list]
            .explode(["NS", "NS_apex", "NS_A", "NS_AAAA"])
            .rename(columns={"NS": "ns", "NS_apex": "ns_apex"}))

def ips_long(col, rtype):
    """1 linha por NS × IP. Se a consulta virou status (str), vai para a coluna 'status'."""
    sub = ns_hosts[["ns", col]].drop_duplicates("ns")
    sub["status"] = sub[col].map(lambda v: "ok" if isinstance(v, list) else v)
    sub["ip"] = sub[col].map(lambda v: v if isinstance(v, list) else [None])
    return (sub.drop(columns=col).explode("ip").assign(rtype=rtype))

ns_ips = pd.concat([ips_long("NS_A", "A"), ips_long("NS_AAAA", "AAAA")], ignore_index=True)

print(dns_status["status"].value_counts())
print(len(ns_hosts), "linhas domínio×NS |", len(ns_ips), "linhas NS×IP")

status
no_ns       18
timeout      9
nxdomain     2
Name: count, dtype: int64
32024 linhas domínio×NS | 26456 linhas NS×IP


In [7]:
companies = df  # já contém a coluna apex
companies.to_parquet("../data/interim/ns_resolver/companies.parquet")
dns_status.to_parquet("../data/interim/ns_resolver/dns_status.parquet")
ns_hosts[["domain", "ns", "ns_apex"]].to_parquet("../data/interim/ns_resolver/ns_hosts.parquet")
ns_ips.to_parquet("../data/interim/ns_resolver/ns_ips.parquet")

In [8]:
(ns_hosts.groupby("ns_apex")["domain"].nunique()
         .sort_values(ascending=False).head(10))

ns_hosts.groupby("ns_apex")["domain"].nunique().to_csv("../data/interim/ns_resolver/ns_apex_counts.csv", index=True)

(ns_hosts.groupby("ns")["domain"].nunique()
         .sort_values(ascending=False).head(10))

ns_hosts.groupby("ns")["domain"].nunique().to_csv("../data/interim/ns_resolver/ns_counts.csv", index=True)

In [9]:
## Número de domínios naõ resolvidos (NXDOMAIN, timeout, invalid...) por status
dns_status["status"].value_counts().to_csv("../data/interim/ns_resolver/dns_status_counts.csv", index=True)
print(dns_status["status"].value_counts())

## numero de dominios resolvidos (sem repetição) 
print("domínios resolvidos:", len(ns_hosts["domain"].unique()))

# lista de faltantes em cada marco (top 500, 1000, 2000 5000 e total)
for n in [500, 1000, 2000, 5000, len(domains)]:
    print(f"top {n}: {len(set(domains[:n]) - set(ns_hosts['domain'].unique()))} faltantes")

status
no_ns       18
timeout      9
nxdomain     2
Name: count, dtype: int64
domínios resolvidos: 10233
top 500: 1 faltantes
top 1000: 3 faltantes
top 2000: 8 faltantes
top 5000: 13 faltantes
top 10261: 29 faltantes
